In [1]:
import faiss
import numpy as np
import time

In [2]:


# --- 1. Setup Data ---
D = 128  # Dimension of the vectors (e.g., from a BERT embedding)
N = 100000  # Number of vectors in the dataset
K = 5    # Number of nearest neighbors to search for

# Generate a synthetic dataset and query vectors
np.random.seed(1234)
xb = np.random.random((N, D)).astype('float32')
xb[:, 0] += np.arange(N) / 1000.  # Add a slight pattern to ensure varied distances
xq = np.random.random((10, D)).astype('float32') # 10 query vectors

# --- 2. IndexHNSWFlat: Graph-Based Index ---

## Initialization and Parameters
M = 32 # The number of links/connections each vector will have (higher M = better accuracy/memory)
efConstruction = 40 # Search complexity during index build (higher = slower build/better accuracy)
efSearch = 16 # Search complexity at query time (higher = slower search/better accuracy)

print("--- HNSW Index Setup ---")
start_time = time.time()
# The HNSW index does not require a separate quantizer or training step
index_hnsw = faiss.IndexHNSWFlat(D, M) 
index_hnsw.hnsw.efConstruction = efConstruction
index_hnsw.hnsw.efSearch = efSearch

# Indexing (Adding)
index_hnsw.add(xb)
print(f"HNSW Build Time: {time.time() - start_time:.3f} seconds")
print(f"HNSW Index Size: {index_hnsw.ntotal} vectors added")

# Searching
start_search_hnsw = time.time()
D_hnsw, I_hnsw = index_hnsw.search(xq, K)
search_time_hnsw = time.time() - start_search_hnsw
print(f"HNSW Search Time (10 queries): {search_time_hnsw:.5f} seconds")
print(f"HNSW Top 5 Distances for Query 1:\n{D_hnsw[0]}")
print("-" * 30)

# --- 3. IndexIVFFlat: Inverted File (Clustering) Index ---

## Initialization and Parameters
nlist = 100  # Number of clusters (Voronoi cells). Rule of thumb: sqrt(N) to 16*sqrt(N).
nprobe = 10  # Number of clusters to search at query time (trade-off: speed vs. accuracy)

print("--- IVF Index Setup ---")
start_time = time.time()
# The quantizer is used to assign vectors to their cluster/list. 
# We use IndexFlatL2 for exact L2 distance computation within the lists.
quantizer = faiss.IndexFlatL2(D) 
index_ivf = faiss.IndexIVFFlat(quantizer, D, nlist, faiss.METRIC_L2)

# Training (Required for IVF)
# This step clusters the vectors and sets the cluster centroids
if not index_ivf.is_trained:
    print("Training IVF Index...")
    index_ivf.train(xb)
    print(f"IVF Training Time: {time.time() - start_time:.3f} seconds")
    
# Indexing (Adding)
index_ivf.add(xb)
print(f"IVF Index Size: {index_ivf.ntotal} vectors added")

# Set the search parameter
index_ivf.nprobe = nprobe 

# Searching
start_search_ivf = time.time()
D_ivf, I_ivf = index_ivf.search(xq, K)
search_time_ivf = time.time() - start_search_ivf
print(f"IVF Search Time (10 queries): {search_time_ivf:.5f} seconds")
print(f"IVF Top 5 Distances for Query 1:\n{D_ivf[0]}")
print("-" * 30)

# --- Comparison ---
# (Note: Actual speed/accuracy comparisons would require a brute-force baseline)
print("--- Performance Comparison ---")
print(f"HNSW Total Search Time: {search_time_hnsw:.5f} s")
print(f"IVF Total Search Time: {search_time_ivf:.5f} s")

--- HNSW Index Setup ---
HNSW Build Time: 2.613 seconds
HNSW Index Size: 100000 vectors added
HNSW Search Time (10 queries): 0.00051 seconds
HNSW Top 5 Distances for Query 1:
[15.291571 15.752586 16.057346 16.204773 16.361649]
------------------------------
--- IVF Index Setup ---
Training IVF Index...
IVF Training Time: 0.551 seconds
IVF Index Size: 100000 vectors added
IVF Search Time (10 queries): 0.01267 seconds
IVF Top 5 Distances for Query 1:
[15.291571 15.752586 16.057346 16.204773 16.361649]
------------------------------
--- Performance Comparison ---
HNSW Total Search Time: 0.00051 s
IVF Total Search Time: 0.01267 s
